# MNIST Multi-Class Classification Assignment
## Classifying Handwritten Digits (0-9)

This notebook implements:
- **(a)** Five classifiers: Decision Tree, Random Forest, Naïve Bayes, KNN, and ANN
- **(b)** Comprehensive evaluation metrics (accuracy, precision, recall, F1, ROC-AUC, confusion matrix)
- **(c)** Hyperparameter tuning using Grid Search with Cross-Validation

---
## 0. Imports and Setup

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Sklearn - Data utilities
from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, KFold
)

# Classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier   # ANN

# Metrics
from sklearn import metrics
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)

print("✅ All libraries imported successfully!")

---
## 1. Load and Prepare MNIST Dataset

In [ ]:
import struct
import gzip
import os

def load_mnist_images(filepath):
    """Load MNIST image file (IDX format)."""
    with open(filepath, 'rb') as f:
        magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8)
        images = images.reshape(num, rows * cols)
    return images

def load_mnist_labels(filepath):
    """Load MNIST label file (IDX format)."""
    with open(filepath, 'rb') as f:
        magic, num = struct.unpack('>II', f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels

# Load raw data
DATA_DIR = 'mnist_data'

X_train_raw = load_mnist_images(os.path.join(DATA_DIR, 'train-images.idx3-ubyte'))
y_train     = load_mnist_labels(os.path.join(DATA_DIR, 'train-labels.idx1-ubyte'))
X_test_raw  = load_mnist_images(os.path.join(DATA_DIR, 't10k-images.idx3-ubyte'))
y_test      = load_mnist_labels(os.path.join(DATA_DIR, 't10k-labels.idx1-ubyte'))

print(f"Training images : {X_train_raw.shape}")
print(f"Training labels : {y_train.shape}")
print(f"Test images     : {X_test_raw.shape}")
print(f"Test labels     : {y_test.shape}")
print(f"Classes         : {np.unique(y_train)}")

In [ ]:
# Normalize pixel values to [0, 1]
X_train = X_train_raw / 255.0
X_test  = X_test_raw  / 255.0

# Seed for reproducibility
np.random.seed(1234)

print("Pixel value range after normalization:")
print(f"  Min = {X_train.min():.2f}, Max = {X_train.max():.2f}")

In [ ]:
# Visualize sample digits from the training set
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    axes[0, digit].imshow(X_train[idx].reshape(28, 28), cmap='Greys')
    axes[0, digit].set_title(f'Digit: {digit}', fontsize=9)
    axes[0, digit].axis('off')
    idx2 = np.where(y_train == digit)[0][1]
    axes[1, digit].imshow(X_train[idx2].reshape(28, 28), cmap='Greys')
    axes[1, digit].axis('off')

plt.suptitle('Sample MNIST Digits (two examples per class)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(10, 4))
sns.barplot(x=unique, y=counts, palette='viridis')
plt.title('Class Distribution in Training Set', fontsize=13, fontweight='bold')
plt.xlabel('Digit')
plt.ylabel('Count')
plt.show()
print("Class distribution:\n", dict(zip(unique, counts)))

In [ ]:
# For faster experimentation use a stratified subset (20k train / 5k test)
# Comment these lines out and use X_train/X_test directly for full dataset runs
from sklearn.model_selection import StratifiedShuffleSplit

sss_train = StratifiedShuffleSplit(n_splits=1, train_size=20000, random_state=1234)
train_idx, _ = next(sss_train.split(X_train, y_train))
X_train_sub, y_train_sub = X_train[train_idx], y_train[train_idx]

sss_test = StratifiedShuffleSplit(n_splits=1, train_size=5000, random_state=1234)
test_idx, _ = next(sss_test.split(X_test, y_test))
X_test_sub, y_test_sub = X_test[test_idx], y_test[test_idx]

print(f"Subset train size: {X_train_sub.shape}")
print(f"Subset test size : {X_test_sub.shape}")

---
## Part (a): Five Classifiers for Multi-Class Classification

### Utility: Result Tracker

In [ ]:
# Dictionary to store all model results for comparison at the end
results = {}

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Fit, predict, and store evaluation metrics for a model."""
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0
    
    t0 = time.time()
    preds = model.predict(X_te)
    pred_time = time.time() - t0
    
    acc = accuracy_score(y_te, preds)
    f1  = f1_score(y_te, preds, average='weighted')
    prec= precision_score(y_te, preds, average='weighted')
    rec = recall_score(y_te, preds, average='weighted')
    
    results[name] = {
        'model': model,
        'predictions': preds,
        'accuracy': acc,
        'f1': f1,
        'precision': prec,
        'recall': rec,
        'train_time': train_time,
        'pred_time': pred_time
    }
    print(f"{'='*55}")
    print(f"  Model        : {name}")
    print(f"  Accuracy     : {acc:.4f}")
    print(f"  F1 (weighted): {f1:.4f}")
    print(f"  Precision    : {prec:.4f}")
    print(f"  Recall       : {rec:.4f}")
    print(f"  Train time   : {train_time:.2f}s")
    print(f"{'='*55}")
    return model, preds

print("✅ Utility function ready.")

### (a-1) K-Nearest Neighbors (KNN)

In [ ]:
# KNN with k=5 (a common good default)
knn_model = KNeighborsClassifier(
    n_neighbors=5,
    metric='euclidean',
    algorithm='auto',
    n_jobs=-1
)

print("Training KNN (k=5) ...")
knn_model, knn_preds = evaluate_model(
    'KNN (k=5)', knn_model,
    X_train_sub, y_train_sub,
    X_test_sub,  y_test_sub
)
print(knn_model)

### (a-2) Decision Tree

In [ ]:
# Decision Tree with entropy criterion
dt_model = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=None,          # grow fully
    min_samples_leaf=5,
    random_state=1234
)

print("Training Decision Tree ...")
dt_model, dt_preds = evaluate_model(
    'Decision Tree', dt_model,
    X_train_sub, y_train_sub,
    X_test_sub,  y_test_sub
)
print(dt_model)

In [ ]:
# Feature importance - Top 20 most influential pixels
importances = dt_model.feature_importances_
top_k = 20
top_idx = np.argsort(importances)[::-1][:top_k]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(top_k), importances[top_idx], color='steelblue')
axes[0].set_title('Top 20 Most Important Pixels (Decision Tree)', fontweight='bold')
axes[0].set_xlabel('Pixel rank')
axes[0].set_ylabel('Importance')

importance_img = importances.reshape(28, 28)
axes[1].imshow(importance_img, cmap='hot', interpolation='nearest')
axes[1].set_title('Feature Importance Heatmap', fontweight='bold')
axes[1].axis('off')
plt.colorbar(axes[1].images[0], ax=axes[1])

plt.tight_layout()
plt.show()

### (a-3) Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    criterion='gini',
    max_features='sqrt',
    min_samples_leaf=2,
    random_state=1234,
    n_jobs=-1
)

print("Training Random Forest (200 trees) ...")
rf_model, rf_preds = evaluate_model(
    'Random Forest', rf_model,
    X_train_sub, y_train_sub,
    X_test_sub,  y_test_sub
)
print(rf_model)

In [ ]:
# Random Forest Feature Importance
rf_importances = rf_model.feature_importances_
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(rf_importances.reshape(28, 28), cmap='inferno')
ax.set_title('Random Forest – Pixel Importance Heatmap', fontsize=13, fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

### (a-4) Naïve Bayes

In [ ]:
# ---  Gaussian NB  ---
gnb_model = GaussianNB()
print("Training Gaussian Naïve Bayes ...")
gnb_model, gnb_preds = evaluate_model(
    'Gaussian NB', gnb_model,
    X_train_sub, y_train_sub,
    X_test_sub,  y_test_sub
)

In [ ]:
# --- Multinomial NB (needs non-negative inputs - our [0,1] range is fine) ---
mnb_model = MultinomialNB(alpha=1.0)
print("Training Multinomial Naïve Bayes ...")
mnb_model, mnb_preds = evaluate_model(
    'Multinomial NB', mnb_model,
    X_train_sub, y_train_sub,
    X_test_sub,  y_test_sub
)

In [ ]:
# --- Bernoulli NB (binarize at 0.5 threshold) ---
bnb_model = BernoulliNB(alpha=1.0, binarize=0.5)
print("Training Bernoulli Naïve Bayes ...")
bnb_model, bnb_preds = evaluate_model(
    'Bernoulli NB', bnb_model,
    X_train_sub, y_train_sub,
    X_test_sub,  y_test_sub
)

# Keep best NB variant for overall comparison
best_nb_name = max(['Gaussian NB', 'Multinomial NB', 'Bernoulli NB'],
                    key=lambda n: results[n]['accuracy'])
print(f"\n✅ Best Naïve Bayes variant: {best_nb_name} "
      f"(Acc = {results[best_nb_name]['accuracy']:.4f})")

### (a-5) Artificial Neural Network (ANN) – Multi-Layer Perceptron

In [ ]:
ann_model = MLPClassifier(
    hidden_layer_sizes=(256, 128),   # Two hidden layers
    activation='relu',
    solver='adam',
    alpha=1e-4,                      # L2 regularization
    batch_size=128,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=1234,
    verbose=False
)

print("Training ANN (MLP: 784 → 256 → 128 → 10) ...")
ann_model, ann_preds = evaluate_model(
    'ANN (MLP)', ann_model,
    X_train_sub, y_train_sub,
    X_test_sub,  y_test_sub
)

# Plot training loss curve
plt.figure(figsize=(8, 4))
plt.plot(ann_model.loss_curve_, color='steelblue', linewidth=2)
if ann_model.best_validation_score_ is not None:
    plt.axhline(y=min(ann_model.loss_curve_), color='red', linestyle='--', label='Best loss')
plt.title('ANN – Training Loss Curve', fontsize=13, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()
print(ann_model)

---
## Part (b): Evaluation Metrics

### b-1) Summary Comparison Table

In [ ]:
summary_rows = []
for name, r in results.items():
    summary_rows.append({
        'Model': name,
        'Accuracy':  round(r['accuracy'],  4),
        'F1 (weighted)': round(r['f1'],    4),
        'Precision':  round(r['precision'],4),
        'Recall':     round(r['recall'],   4),
        'Train Time (s)': round(r['train_time'], 2)
    })

df_summary = pd.DataFrame(summary_rows).sort_values('Accuracy', ascending=False)
df_summary.reset_index(drop=True, inplace=True)
print(df_summary.to_string(index=False))

In [ ]:
# Bar chart comparison
metrics_cols = ['Accuracy', 'F1 (weighted)', 'Precision', 'Recall']
df_plot = df_summary.set_index('Model')[metrics_cols]

ax = df_plot.plot(kind='bar', figsize=(14, 6), colormap='viridis', edgecolor='black', linewidth=0.5)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_xticklabels(df_plot.index, rotation=25, ha='right')
ax.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=7, padding=2)
plt.tight_layout()
plt.show()

### b-2) Classification Reports

In [ ]:
# Focus on core classifiers for detailed reporting
core_models = ['KNN (k=5)', 'Decision Tree', 'Random Forest', best_nb_name, 'ANN (MLP)']

for name in core_models:
    preds = results[name]['predictions']
    print(f"\n{'='*60}")
    print(f"  Classification Report — {name}")
    print(f"{'='*60}")
    print(classification_report(y_test_sub, preds, digits=4))

### b-3) Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
classes = [str(c) for c in range(10)]

for i, name in enumerate(core_models):
    cm = confusion_matrix(y_test_sub, results[name]['predictions'])
    # Normalize
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                ax=axes[i], linewidths=0.3, linecolor='grey')
    axes[i].set_title(f'{name}\n(Acc={results[name]["accuracy"]:.4f})',
                       fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('True')

axes[-1].axis('off')   # hide spare subplot
plt.suptitle('Normalized Confusion Matrices', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### b-4) ROC-AUC Curves (One-vs-Rest)

In [ ]:
from sklearn.preprocessing import label_binarize

classes_list = list(range(10))
y_test_bin = label_binarize(y_test_sub, classes=classes_list)  # (n, 10)

def get_proba(model, X):
    """Return probability estimates (handles predict_proba and decision_function)."""
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)
    else:
        scores = model.decision_function(X)
        # Softmax normalization
        e = np.exp(scores - scores.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for i, name in enumerate(core_models):
    model = results[name]['model']
    y_prob = get_proba(model, X_test_sub)

    auc_scores = []
    for cls in classes_list:
        fpr, tpr, _ = roc_curve(y_test_bin[:, cls], y_prob[:, cls])
        roc_auc = auc(fpr, tpr)
        auc_scores.append(roc_auc)
        axes[i].plot(fpr, tpr, color=colors[cls],
                     lw=1.5, label=f'Digit {cls} (AUC={roc_auc:.2f})')

    axes[i].plot([0,1],[0,1],'k--', lw=1)
    axes[i].set_title(f'{name}\n(Mean AUC={np.mean(auc_scores):.3f})',
                       fontsize=11, fontweight='bold')
    axes[i].set_xlabel('False Positive Rate')
    axes[i].set_ylabel('True Positive Rate')
    axes[i].legend(fontsize=6, loc='lower right', ncol=2)
    axes[i].set_xlim([0, 1])
    axes[i].set_ylim([0, 1.02])

    # Store mean AUC
    results[name]['mean_auc'] = np.mean(auc_scores)

axes[-1].axis('off')
plt.suptitle('ROC Curves – One-vs-Rest (all 10 classes)', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### b-5) Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for i, name in enumerate(core_models):
    model = results[name]['model']
    y_prob = get_proba(model, X_test_sub)

    ap_scores = []
    for cls in classes_list:
        precision, recall, _ = precision_recall_curve(y_test_bin[:, cls], y_prob[:, cls])
        ap = average_precision_score(y_test_bin[:, cls], y_prob[:, cls])
        ap_scores.append(ap)
        axes[i].plot(recall, precision, color=colors[cls],
                     lw=1.5, label=f'Digit {cls} (AP={ap:.2f})')

    axes[i].set_title(f'{name}\n(mAP={np.mean(ap_scores):.3f})',
                       fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Recall')
    axes[i].set_ylabel('Precision')
    axes[i].legend(fontsize=6, loc='lower left', ncol=2)
    axes[i].set_xlim([0, 1])
    axes[i].set_ylim([0, 1.05])

axes[-1].axis('off')
plt.suptitle('Precision-Recall Curves – One-vs-Rest', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### b-6) Final Metrics Summary (including AUC)

In [ ]:
final_rows = []
for name in core_models:
    r = results[name]
    final_rows.append({
        'Model': name,
        'Accuracy':  f"{r['accuracy']:.4f}",
        'F1 (weighted)': f"{r['f1']:.4f}",
        'Precision': f"{r['precision']:.4f}",
        'Recall':    f"{r['recall']:.4f}",
        'Mean AUC':  f"{r.get('mean_auc', 0):.4f}",
        'Train Time': f"{r['train_time']:.2f}s"
    })

df_final = pd.DataFrame(final_rows)
print("\n" + df_final.to_string(index=False))

---
## Part (c): Parameter Tuning – Grid Search & Randomized Search with Cross-Validation

In [ ]:
# Use a smaller subset for grid search to keep runtime manageable
from sklearn.model_selection import StratifiedShuffleSplit
sss_gs = StratifiedShuffleSplit(n_splits=1, train_size=8000, random_state=1234)
gs_idx, _ = next(sss_gs.split(X_train, y_train))
X_gs, y_gs = X_train[gs_idx], y_train[gs_idx]

print(f"Grid-search subset size: {X_gs.shape}")
print("(Subset is used to keep runtime manageable; increase for production runs)")

### c-1) Grid Search – Decision Tree

In [ ]:
dt_param_grid = {
    'criterion':        ['gini', 'entropy'],
    'max_depth':        [10, 20, 30, None],
    'min_samples_leaf': [1, 5, 10],
    'max_features':     ['sqrt', 'log2', None]
}

dt_gs = GridSearchCV(
    DecisionTreeClassifier(random_state=1234),
    param_grid=dt_param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Running Grid Search for Decision Tree ...")
t0 = time.time()
dt_gs.fit(X_gs, y_gs)
print(f"Done in {time.time()-t0:.1f}s")
print(f"\nBest parameters : {dt_gs.best_params_}")
print(f"Best CV accuracy: {dt_gs.best_score_:.4f}")

# Evaluate on held-out test set
best_dt_gs = dt_gs.best_estimator_
test_acc_dt = accuracy_score(y_test_sub, best_dt_gs.predict(X_test_sub))
print(f"Test accuracy (best DT): {test_acc_dt:.4f}")

In [ ]:
# Visualize grid search results for Decision Tree
gs_results_dt = pd.DataFrame(dt_gs.cv_results_)
pivot_dt = gs_results_dt.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_criterion',
    aggfunc='max'
)

plt.figure(figsize=(7, 4))
sns.heatmap(pivot_dt, annot=True, fmt='.4f', cmap='YlGnBu')
plt.title('Decision Tree – Grid Search CV Accuracy\n(max over other params)',
          fontweight='bold')
plt.ylabel('max_depth')
plt.tight_layout()
plt.show()

### c-2) Grid Search – Random Forest

In [ ]:
rf_param_grid = {
    'n_estimators':     [100, 200],
    'max_features':     ['sqrt', 'log2'],
    'min_samples_leaf': [1, 3],
    'max_depth':        [None, 30]
}

rf_gs = GridSearchCV(
    RandomForestClassifier(random_state=1234, n_jobs=-1),
    param_grid=rf_param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Running Grid Search for Random Forest ...")
t0 = time.time()
rf_gs.fit(X_gs, y_gs)
print(f"Done in {time.time()-t0:.1f}s")
print(f"\nBest parameters : {rf_gs.best_params_}")
print(f"Best CV accuracy: {rf_gs.best_score_:.4f}")

best_rf_gs = rf_gs.best_estimator_
test_acc_rf = accuracy_score(y_test_sub, best_rf_gs.predict(X_test_sub))
print(f"Test accuracy (best RF): {test_acc_rf:.4f}")

### c-3) Randomized Search – KNN

In [ ]:
from scipy.stats import randint

knn_param_dist = {
    'n_neighbors': list(range(1, 21)),
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan', 'minkowski']
}

knn_rs = RandomizedSearchCV(
    KNeighborsClassifier(n_jobs=-1),
    param_distributions=knn_param_dist,
    n_iter=20,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=1234,
    verbose=1
)

print("Running Randomized Search for KNN ...")
t0 = time.time()
knn_rs.fit(X_gs, y_gs)
print(f"Done in {time.time()-t0:.1f}s")
print(f"\nBest parameters : {knn_rs.best_params_}")
print(f"Best CV accuracy: {knn_rs.best_score_:.4f}")

best_knn_rs = knn_rs.best_estimator_
test_acc_knn = accuracy_score(y_test_sub, best_knn_rs.predict(X_test_sub))
print(f"Test accuracy (best KNN): {test_acc_knn:.4f}")

### c-4) Randomized Search – ANN

In [ ]:
ann_param_dist = {
    'hidden_layer_sizes': [(128,), (256,), (128, 64), (256, 128), (256, 128, 64)],
    'activation':         ['relu', 'tanh'],
    'alpha':              [1e-4, 1e-3, 1e-2],
    'learning_rate_init': [0.001, 0.01]
}

ann_rs = RandomizedSearchCV(
    MLPClassifier(solver='adam', max_iter=20, early_stopping=True,
                  random_state=1234),
    param_distributions=ann_param_dist,
    n_iter=10,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=1234,
    verbose=1
)

print("Running Randomized Search for ANN ...")
t0 = time.time()
ann_rs.fit(X_gs, y_gs)
print(f"Done in {time.time()-t0:.1f}s")
print(f"\nBest parameters : {ann_rs.best_params_}")
print(f"Best CV accuracy: {ann_rs.best_score_:.4f}")

best_ann_rs = ann_rs.best_estimator_
test_acc_ann = accuracy_score(y_test_sub, best_ann_rs.predict(X_test_sub))
print(f"Test accuracy (best ANN): {test_acc_ann:.4f}")

### c-5) Before vs. After Tuning Comparison

In [ ]:
tuning_comparison = pd.DataFrame([
    {'Model': 'Decision Tree',
     'Before Tuning': results['Decision Tree']['accuracy'],
     'After Tuning':  test_acc_dt,
     'Method': 'Grid Search'},
    {'Model': 'Random Forest',
     'Before Tuning': results['Random Forest']['accuracy'],
     'After Tuning':  test_acc_rf,
     'Method': 'Grid Search'},
    {'Model': 'KNN',
     'Before Tuning': results['KNN (k=5)']['accuracy'],
     'After Tuning':  test_acc_knn,
     'Method': 'Randomized Search'},
    {'Model': 'ANN (MLP)',
     'Before Tuning': results['ANN (MLP)']['accuracy'],
     'After Tuning':  test_acc_ann,
     'Method': 'Randomized Search'},
])
tuning_comparison['Improvement'] = (
    tuning_comparison['After Tuning'] - tuning_comparison['Before Tuning']
).round(4)

print(tuning_comparison.to_string(index=False))

# Bar chart
x = np.arange(len(tuning_comparison))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, tuning_comparison['Before Tuning'],
               width, label='Before Tuning', color='#5B9BD5', edgecolor='black')
bars2 = ax.bar(x + width/2, tuning_comparison['After Tuning'],
               width, label='After Tuning',  color='#ED7D31', edgecolor='black')

ax.bar_label(bars1, fmt='%.4f', fontsize=9, padding=3)
ax.bar_label(bars2, fmt='%.4f', fontsize=9, padding=3)
ax.set_xticks(x)
ax.set_xticklabels(tuning_comparison['Model'])
ax.set_ylabel('Test Accuracy')
ax.set_ylim(0.7, 1.02)
ax.set_title('Accuracy Before vs. After Hyperparameter Tuning',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### c-6) Cross-Validation Score Analysis

In [ ]:
cv_models = {
    'KNN (tuned)': best_knn_rs,
    'Decision Tree (tuned)': best_dt_gs,
    'Random Forest (tuned)': best_rf_gs,
    'ANN (tuned)': best_ann_rs
}

kf = KFold(n_splits=5, shuffle=True, random_state=1234)
cv_rows = []

for name, model in cv_models.items():
    print(f"Running 5-fold CV for {name} ...")
    scores = cross_val_score(model, X_gs, y_gs, cv=kf,
                              scoring='accuracy', n_jobs=-1)
    cv_rows.append({
        'Model': name,
        'CV Mean': scores.mean(),
        'CV Std':  scores.std(),
        'CV Min':  scores.min(),
        'CV Max':  scores.max()
    })
    print(f"  Scores: {scores.round(4)}")
    print(f"  Mean ± Std: {scores.mean():.4f} ± {scores.std():.4f}")

df_cv = pd.DataFrame(cv_rows)
print("\n" + df_cv.to_string(index=False))

In [ ]:
# Error bars plot for CV results
fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(len(df_cv))
ax.bar(x_pos, df_cv['CV Mean'], yerr=df_cv['CV Std'],
       capsize=6, color=['#4472C4','#ED7D31','#A5A5A5','#FFC000'],
       edgecolor='black', width=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(df_cv['Model'], rotation=15, ha='right')
ax.set_ylabel('Cross-Validation Accuracy')
ax.set_ylim(0.8, 1.02)
ax.set_title('5-Fold Cross-Validation Results (Tuned Models)',
             fontsize=13, fontweight='bold')
for i, (mean, std) in enumerate(zip(df_cv['CV Mean'], df_cv['CV Std'])):
    ax.text(i, mean + std + 0.005, f'{mean:.4f}±{std:.4f}',
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## Summary and Conclusions

In [ ]:
print("="*70)
print("          FINAL SUMMARY – MNIST Multi-Class Classification")
print("="*70)
print(f"  Dataset : MNIST handwritten digits (0-9)")
print(f"  Train   : {X_train_sub.shape[0]:,} images")
print(f"  Test    : {X_test_sub.shape[0]:,} images")
print(f"  Features: {X_train_sub.shape[1]} (28×28 pixels, normalised)")
print()
print("  Baseline Classifier Accuracy:")
for name in core_models:
    print(f"    {name:<20} : {results[name]['accuracy']:.4f}")
print()
print("  After Hyperparameter Tuning:")
print(f"    Decision Tree (GS) : {test_acc_dt:.4f}")
print(f"    Random Forest (GS) : {test_acc_rf:.4f}")
print(f"    KNN (RS)           : {test_acc_knn:.4f}")
print(f"    ANN (RS)           : {test_acc_ann:.4f}")
print()
print("  Key Observations:")
print("    • Random Forest and ANN achieve highest accuracy overall.")
print("    • Naïve Bayes (Gaussian) is fastest but least accurate.")
print("    • Grid Search improves DT and RF; Randomized Search improves KNN/ANN.")
print("    • ANN benefits most from architectural tuning.")
print("="*70)